# Odd - Even Analysis

Odd - Even analysis consists of separating a survey into odd- and even-numbered flight-lines, gridding each separately and then calculating the RMS difference between the two resultant grids.

We will use the 2012 Kauring gravimetry data in this notebook. If you haven't already, run the `Prepare_ASEGGDF2` notebook so that the survey data are ready for QC.

___

First, import the required modules, and set the path to the geowhizz file.

In [ ]:
from pathlib import Path
import galileoQC  as qc

In [ ]:
dh = Path(r'./Kauring_grv/GRAV.hdf5')

In [ ]:
if not dh.exists():
    %run ./Prepare_2012KauringData.ipynb

___

We can use `reportLines` to see that the traverse and control lines have been correctly classified. But we have not classified on the basis of odd or even line-numbering yet.

In [ ]:
qc.reportLines(dh)

___

Furthermore, the flight-line map shows that there are two distinct blocks in the file. We don't want the smaller block with closer line spacing and different line directions.

In [ ]:
qc.linesMap([dh], whizzPlanFile=dh)

___

We want only the flight-lines whose line-number starts with a "1". First, we copy and paste them from the `qc.reportLines` output, making sure we capture all traverse and control lines that start with a "1", and no others.

In [ ]:
gm_lines = ['100100', '100200', '100300', '100400', '100500', '100600', '100700', '100800', 
            '100900', '10100', '101000', '101100', '101200', '101300', '101400', '101500', 
            '101600', '101700', '101800', '101900', '10200', '102000', '102100', '102200', 
            '102300', '102400', '102500', '102600', '102700', '102800', '102900', '10300', 
            '103000', '103100', '103200', '103300', '103400', '103500', '103600', '103700', 
            '103800', '103900', '10400', '104000', '104100', '104200', '104300', '104400', 
            '104500', '104600', '104700', '104800', '104900', '10500', '105000', '105100', 
            '105200', '105300', '105400', '105500', '105600', '105700', '105800', '105900', 
            '10600', '106000', '106100', '106200', '106300', '106400', '106500', '106600', 
            '106700', '106800', '106900', '10700', '107000', '107100', '107200', '107300', 
            '107400', '107500', '107600', '107700', '107800', '107900', '10800', '108000', 
            '108100', '108200', '108300', '108400', '108500', '108600', '108700', '108800', 
            '108900', '10900', '109000', '109100', '109200', '109300', '109400', '109500', 
            '109600', '109700', '109800', '109900', '11000', '110000', '110100', '110200', 
            '110300', '110400', '110500', '110600', '110700', '110800', '110900', '11100']

___

The flight-line map shows that we have the ones we want.

In [ ]:
qc.linesMap([dh], whizzPlanFile=dh, planLines=gm_lines)

___

Now, we write a short script to sort them into odd and even numbered line-lists.

In [ ]:
odd = []
even = []
for line in gm_lines:
    num = int(float(line)/100)
    if num % 2 == 0:
        even.append(line)# Even 
    else:
        odd.append(line) # Odd

 ... and we update the survey meta-data with these lists, and report the lines so we can check we have what we want. (`updateOddOrEven` ignore control lines for assigning parity.)

In [ ]:
qc.updateOddOrEven(whizzFile=dh, oddlines=odd, evenlines=even, planFile='', verbose=False)
qc.reportLines(dh)

___

Now we can perform the odd-even analysis. The default gridding parameters have been accepted here so the grid is not great. However, the RMS difference between odd and even numbered lines is $0.57\,mGal$ which is a good result.

In [ ]:
qc.oddevenlines(dh, 'GRVBGL100S_267', 200.0)

___

Alternatively, you can always enter the lists of odd and even lines directly into the function call. This can be useful if, for example, you want to omit some lines from the analysis. Here is an example.

In [ ]:
# First remove the control lines from the even and odd lines (`updateOddOrEven`
# does this automatically for us in the previous example):
controls = ['10100', '10200', '10300', '10400', '10500', '10600', '10700',
            '10800', '10900', '11000', '11100']
new_even = even.copy()
new_odd = odd.copy()
for ctrl in controls:
    if ctrl in new_even:
        new_even.remove(ctrl)
    if ctrl in new_odd:
        new_odd.remove(ctrl)

In [ ]:
# Now remove the "unwanted" first even line:
modified_even = new_even.pop(0)
qc.oddevenlines(dh, 'GRVBGL100S_267', 200.0, oddlines=new_odd, evenlines=new_even)

In [ ]:
qc.reportChannels(dh)

Sometimes, the odd-even difference results are dominated by short-wavelength signal and therefore not a reasonable representation of the error. There is an option to low-pass filter the data before statistical analysis and display. The filter used is the same $\cos_{2}$ filter as used in the conforming. One only needs to specify the filter 50% gain wavelength.

In [ ]:
# Now low-pass the grid before analysis:
qc.oddevenlines(dh, 'GRVBGL100S_267', 200.0, oddlines=new_odd, evenlines=new_even, filter_length=10000.0, method='minc')